# RAG Prototype — Question 1 Code Submission





In [1]:
# uncomment to install (only needs to be run once)
!pip install scikit-learn sentence-transformers anthropic

In [2]:
import re
import os
import math
import random
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    from sentence_transformers import SentenceTransformer, CrossEncoder
    HAVE_SBERT = True
except ImportError:
    HAVE_SBERT = False

try:
    import anthropic
    HAVE_ANTHROPIC_LIB = True
except ImportError:
    HAVE_ANTHROPIC_LIB = False

print("scikit-learn: ok")
print("sentence-transformers available:", HAVE_SBERT)
print("anthropic library available:", HAVE_ANTHROPIC_LIB)
print("ANTHROPIC_API_KEY set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

scikit-learn: ok
sentence-transformers available: True
anthropic library available: True
ANTHROPIC_API_KEY set: False


In [3]:
docs = [
    {
        "id": "CBI-OUT-2021",
        "title": "Cross-Industry Guidance on Outsourcing",
        "source": "Central Bank of Ireland, December 2021 - centralbank.ie/docs/default-source/publications/consultation-papers/cp138/cross-industry-guidance-on-outsourcing.pdf",
        "date": "2021-12-17",
        "current": True,
        "public": True,
        "sections": {
            "P13": (
                "Notify the Central Bank of planned critical or important outsourcing "
                "arrangements and of material changes to existing critical or important "
                "outsourcing arrangements. The Central Bank has clarified that notification "
                "of such proposed arrangements does not constitute a pre-approval process "
                "and specific timings in respect of the submission of notifications are not "
                "prescribed unless required by existing regulation."
            ),
            "P14": (
                "Develop and maintain an outsourcing register to include prescribed "
                "information for all existing and future outsourcing arrangements."
            ),
        }
    },
    {
        # this is the CONSULTATION PAPER that came before the guidance above -
        # a real, genuinely-superseded document, used to test the "exclude
        # non-current documents" filter
        "id": "CBI-CP138-2021",
        "title": "CP138 - Consultation on Cross-Industry Guidance on Outsourcing",
        "source": "Central Bank of Ireland - centralbank.ie/publication/consultation-papers/cp138",
        "date": "2021-01-01",
        "current": False,   # superseded by the finalised Guidance (CBI-OUT-2021) above
        "public": True,
        "sections": {
            "1.1": (
                "The Cross-Industry Guidance on Outsourcing outlines the Central Bank of "
                "Ireland's expectations regarding the management of outsourcing risk, with "
                "a view to promoting higher standards of operational resilience in "
                "regulated financial service providers."
            ),
        }
    },
    {
        "id": "EU-DORA-2022",
        "title": "Regulation (EU) 2022/2554 (DORA), Article 19 - reporting of major ICT-related incidents",
        "source": "EUR-Lex, OJ L 333, 27.12.2022; timings per Commission Delegated Regulation (EU) 2025/301, Art. 5",
        "date": "2025-01-17",
        "current": True,
        "public": True,
        "sections": {
            "Art19": (
                "Financial entities shall report major ICT-related incidents to the "
                "relevant competent authority. The initial notification shall be "
                "submitted as early as possible, in any case within four hours from "
                "classification of the incident as major, and no later than 24 hours "
                "from the moment the financial entity became aware of the incident. "
                "The intermediate report shall be submitted at the latest within 72 "
                "hours from the submission of the initial notification. The final "
                "report shall be submitted no later than one month after the "
                "intermediate report."
            ),
        }
    },
    {
        # NOT a real document - invented to represent what a bank's internal
        # ICT policy might contain, since no real one is publicly available.
        # kept clearly separate from the real documents above.
        "id": "INT-POL-ICT-07",
        "title": "[ILLUSTRATIVE, NOT A REAL DOCUMENT] Internal Policy: ICT Third-Party Risk",
        "source": "invented for this project - represents a fictional bank's internal policy",
        "date": "2024-06-01",
        "current": True,
        "public": False,     # internal only - used to test access control
        "sections": {
            "5.2": (
                "Any outsourcing arrangement with an annual contract value exceeding "
                "EUR 250,000 must be escalated to the Group Outsourcing Committee "
                "before contract signature."
            ),
            "6.1": (
                "An ICT incident is classified as major if it affects more than 10 "
                "percent of active retail customers or causes an outage exceeding 2 hours."
            ),
        }
    },
    {
        "id": "CBI-FP-S21",
        "title": "Fitness and Probity - Central Bank Reform Act 2010, s.21(1) and CBI Guidance",
        "source": "Central Bank Reform Act 2010, s.21(1); Guidance on the Standards of Fitness and Probity, centralbank.ie",
        "date": "2025-11-20",
        "current": True,
        "public": True,
        "sections": {
            "s21": (
                "A regulated financial service provider shall not permit a person to "
                "perform a controlled function unless the regulated financial service "
                "provider is satisfied on reasonable grounds that the person complies "
                "with any standard of fitness and probity in a code issued under "
                "section 50, and the person has agreed to comply with any such standard."
            ),
            "monitoring": (
                "Firms are required to monitor on an ongoing basis the fitness and "
                "probity of persons performing controlled functions, and are expected "
                "to carry out this review on at least an annual basis."
            ),
        }
    }
]

print(len(docs), "documents loaded")
for d in docs:
    tag = "REAL" if "ILLUSTRATIVE" not in d["title"] else "SYNTHETIC (illustrative only)"
    print(f" - [{tag}] {d['id']}: {d['title']}")

5 documents loaded
 - [REAL] CBI-OUT-2021: Cross-Industry Guidance on Outsourcing
 - [REAL] CBI-CP138-2021: CP138 - Consultation on Cross-Industry Guidance on Outsourcing
 - [REAL] EU-DORA-2022: Regulation (EU) 2022/2554 (DORA), Article 19 - reporting of major ICT-related incidents
 - [SYNTHETIC (illustrative only)] INT-POL-ICT-07: [ILLUSTRATIVE, NOT A REAL DOCUMENT] Internal Policy: ICT Third-Party Risk
 - [REAL] CBI-FP-S21: Fitness and Probity - Central Bank Reform Act 2010, s.21(1) and CBI Guidance


In [4]:
def make_chunks(docs):
    chunks = []
    for doc in docs:
        for section_no, text in doc["sections"].items():
            chunk = {
                "id": doc["id"] + "::" + section_no,
                "doc_id": doc["id"],
                "section": section_no,
                "text": "[" + doc["title"] + " " + section_no + "] " + text,
                "raw_text": text,
                "current": doc["current"],
                "public": doc["public"],
                "date": doc["date"]
            }
            chunks.append(chunk)
    return chunks


def make_chunks_fixed_width(docs, width=25):
    chunks = []
    for doc in docs:
        words = []
        for text in doc["sections"].values():
            words += text.split()
        i = 0
        n = 0
        while i < len(words):
            piece = " ".join(words[i:i+width])
            chunks.append({
                "id": doc["id"] + "::fw" + str(n),
                "doc_id": doc["id"],
                "section": "n/a",
                "text": piece,
                "raw_text": piece,
                "current": doc["current"],
                "public": doc["public"],
                "date": doc["date"]
            })
            i += width
            n += 1
    return chunks


def filter_chunks(chunks, allow_internal=True, current_only=True):
    out = []
    for c in chunks:
        if current_only and not c["current"]:
            continue
        if not allow_internal and not c["public"]:
            continue
        out.append(c)
    return out


chunks = filter_chunks(make_chunks(docs), allow_internal=True, current_only=True)
print(len(chunks), "chunks indexed (current + internal access)")
for c in chunks:
    print(" -", c["id"])

7 chunks indexed (current + internal access)
 - CBI-OUT-2021::P13
 - CBI-OUT-2021::P14
 - EU-DORA-2022::Art19
 - INT-POL-ICT-07::5.2
 - INT-POL-ICT-07::6.1
 - CBI-FP-S21::s21
 - CBI-FP-S21::monitoring


In [5]:
def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


class BM25:
    def __init__(self, texts, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.docs = [tokenize(t) for t in texts]
        self.n_docs = len(self.docs)
        self.doc_lens = [len(d) for d in self.docs]
        self.avg_len = sum(self.doc_lens) / self.n_docs
        self.term_freqs = [Counter(d) for d in self.docs]

        df = Counter()
        for d in self.docs:
            for term in set(d):
                df[term] += 1
        self.idf = {}
        for term, n in df.items():
            self.idf[term] = math.log(1 + (self.n_docs - n + 0.5) / (n + 0.5))

    def score(self, query):
        q_terms = tokenize(query)
        scores = [0.0] * self.n_docs
        for term in q_terms:
            if term not in self.idf:
                continue
            idf = self.idf[term]
            for i in range(self.n_docs):
                f = self.term_freqs[i].get(term, 0)
                if f == 0:
                    continue
                denom = f + self.k1 * (1 - self.b + self.b * self.doc_lens[i] / self.avg_len)
                scores[i] += idf * f * (self.k1 + 1) / denom
        return scores

In [6]:
# only try to load the real embedding model once, not once per retriever
_sbert_model = None
_sbert_tried = False


def _get_sbert():
    global _sbert_model, _sbert_tried
    if not _sbert_tried:
        _sbert_tried = True
        if HAVE_SBERT:
            try:
                _sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
                print("dense retrieval: using all-MiniLM-L6-v2 (sentence-transformers)")
            except Exception as e:
                print(f"dense retrieval: could not load embedding model ({type(e).__name__}) "
                      "- falling back to TF-IDF")
        else:
            print("dense retrieval: sentence-transformers not installed - using TF-IDF")
    return _sbert_model


class DenseSearch:
    def __init__(self, texts):
        self.texts = texts
        self.model = _get_sbert()
        if self.model is not None:
            self.embeddings = self.model.encode(texts)
        else:
            self.vec = TfidfVectorizer(stop_words="english")
            self.matrix = self.vec.fit_transform(texts)

    def score(self, query):
        if self.model is not None:
            q_emb = self.model.encode([query])
            sims = cosine_similarity(q_emb, self.embeddings)[0]
        else:
            q_vec = self.vec.transform([query])
            sims = cosine_similarity(q_vec, self.matrix)[0]
        return list(sims)


def rank_from_scores(scores):
    order = sorted(range(len(scores)), key=lambda i: -scores[i])
    ranks = [0] * len(scores)
    for rank, idx in enumerate(order, start=1):
        ranks[idx] = rank
    return ranks


class HybridRetriever:
    def __init__(self, chunks, w_sparse=0.5, rrf_k=60):
        self.chunks = chunks
        texts = [c["text"] for c in chunks]
        self.bm25 = BM25(texts)
        self.vecs = DenseSearch(texts)
        self.w_sparse = w_sparse
        self.rrf_k = rrf_k

    def search(self, query, top_k=10):
        sparse_scores = self.bm25.score(query)
        dense_scores = self.vecs.score(query)
        sparse_ranks = rank_from_scores(sparse_scores)
        dense_ranks = rank_from_scores(dense_scores)

        combined = []
        for i in range(len(self.chunks)):
            fused = (self.w_sparse / (self.rrf_k + sparse_ranks[i]) +
                     (1 - self.w_sparse) / (self.rrf_k + dense_ranks[i]))
            combined.append((self.chunks[i], fused))

        combined.sort(key=lambda x: -x[1])
        return combined[:top_k]


retriever = HybridRetriever(chunks)
print("retriever built")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

dense retrieval: using all-MiniLM-L6-v2 (sentence-transformers)
retriever built


In [7]:
try:
    _cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2") if HAVE_SBERT else None
    if _cross_encoder is not None:
        print("reranker: using cross-encoder/ms-marco-MiniLM-L-6-v2")
    else:
        print("reranker: sentence-transformers not installed - falling back to word overlap")
except Exception as e:
    _cross_encoder = None
    print(f"reranker: could not load cross-encoder ({e}) - falling back to word overlap")


def rerank(query, candidates, top_n=5):
    if _cross_encoder is not None:
        pairs = [(query, c["text"]) for c, _ in candidates]
        scores = _cross_encoder.predict(pairs)
        scored = [(candidates[i][0], float(scores[i])) for i in range(len(candidates))]
    else:
        q_words = set(tokenize(query))
        scored = []
        for chunk, old_score in candidates:
            words = tokenize(chunk["text"])
            overlap = len(q_words.intersection(words))
            score = overlap / max(1, len(q_words))
            scored.append((chunk, score))

    scored.sort(key=lambda x: -x[1])
    return scored[:top_n]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

reranker: using cross-encoder/ms-marco-MiniLM-L-6-v2


In [8]:
USE_LLM = HAVE_ANTHROPIC_LIB and bool(os.environ.get("ANTHROPIC_API_KEY"))
if USE_LLM:
    print("generation: using Claude API")
else:
    print("generation: no ANTHROPIC_API_KEY set - falling back to sentence extraction")


def word_overlap(query, text):
    q = set(w for w in tokenize(query) if len(w) > 3)
    if not q:
        return 0
    t = set(tokenize(text))
    return len(q.intersection(t)) / len(q)


def call_llm(query, passages):
    client = anthropic.Anthropic()
    numbered = "\n".join(f"[{i}] {p['raw_text']}" for i, p in enumerate(passages, start=1))
    prompt = (
        "Answer the question using ONLY the numbered passages below. "
        "Put a [n] citation after every fact you use, matching the passage number. "
        "If the passages don't contain the answer, reply exactly: INSUFFICIENT_EVIDENCE\n\n"
        f"PASSAGES:\n{numbered}\n\nQUESTION: {query}\nANSWER:"
    )
    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=400,
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )
    return "".join(block.text for block in resp.content if block.type == "text")


def generate_answer(query, passages, abstain_threshold=0.3):
    if not passages:
        return {"text": "INSUFFICIENT_EVIDENCE - nothing was retrieved.",
                "cites": [], "abstained": True}

    best_match = max(word_overlap(query, p["raw_text"]) for p in passages)
    if best_match < abstain_threshold:
        return {"text": "INSUFFICIENT_EVIDENCE - none of the retrieved passages "
                         "look relevant enough to answer this.",
                "cites": [], "abstained": True}

    if USE_LLM:
        text = call_llm(query, passages)
        cite_nums = set(int(n) for n in re.findall(r"\[(\d+)\]", text))
        cites = [passages[n - 1]["id"] for n in cite_nums if 0 < n <= len(passages)]
        abstained = text.strip().startswith("INSUFFICIENT_EVIDENCE")
        return {"text": text, "cites": cites, "abstained": abstained}

    scored_sentences = []
    for i, p in enumerate(passages, start=1):
        for sent in re.split(r"(?<=\.)\s+", p["raw_text"]):
            if len(sent.split()) < 5:
                continue
            scored_sentences.append((word_overlap(query, sent), i, sent))

    scored_sentences.sort(key=lambda x: -x[0])
    top = [s for s in scored_sentences[:3] if s[0] > 0]

    if not top:
        return {"text": "INSUFFICIENT_EVIDENCE", "cites": [], "abstained": True}

    text = " ".join(f"{sent} [{i}]" for _, i, sent in top)
    cites = [passages[i - 1]["id"] for _, i, _ in top]
    return {"text": text, "cites": cites, "abstained": False}


def verify_answer(answer, passages):
    if answer["abstained"]:
        return answer

    sentences = re.findall(r"(.+?)\s*\[(\d+)\]", answer["text"])
    good = []
    for sent, idx in sentences:
        idx = int(idx)
        if idx - 1 >= len(passages):
            continue
        source = passages[idx - 1]["raw_text"]
        claim_numbers = set(re.findall(r"\d+", sent))
        source_numbers = set(re.findall(r"\d+", source))
        if claim_numbers.issubset(source_numbers):
            good.append(f"{sent.strip()} [{idx}]")

    if not good:
        return {"text": "INSUFFICIENT_EVIDENCE - could not verify the numbers "
                         "in the draft answer against the sources.",
                "cites": [], "abstained": True}

    return {"text": " ".join(good), "cites": answer["cites"], "abstained": False}


def answer_question(retriever, query, use_reranker=True, use_verifier=True):
    candidates = retriever.search(query, top_k=10)
    if use_reranker:
        top_passages = [c for c, s in rerank(query, candidates, top_n=5)]
    else:
        top_passages = [c for c, s in candidates[:5]]

    answer = generate_answer(query, top_passages)
    if use_verifier:
        answer = verify_answer(answer, top_passages)
    return answer, top_passages

print("pipeline functions ready")

generation: no ANTHROPIC_API_KEY set - falling back to sentence extraction
pipeline functions ready


In [9]:
#Run this cell with any question about the corpus. Change `my_question`below and re-run to try your own.
my_question = "What must a regulated firm notify the Central Bank about regarding outsourcing arrangements?"

answer, passages_used = answer_question(retriever, my_question)

print("Q:", my_question)
print("A:", answer["text"])
print()
print("cited chunks:", answer["cites"])
print("abstained:", answer["abstained"])

Q: What must a regulated firm notify the Central Bank about regarding outsourcing arrangements?
A: Notify the Central Bank of planned critical or important outsourcing arrangements and of material changes to existing critical or important outsourcing arrangements. [1] The Central Bank has clarified that notification of such proposed arrangements does not constitute a pre-approval process and specific timings in respect of the submission of notifications are not prescribed unless required by existing regulation. [1] Any outsourcing arrangement with an annual contract value exceeding EUR 250,000 must be escalated to the Group Outsourcing Committee before contract signature. [2]

cited chunks: ['CBI-OUT-2021::P13', 'CBI-OUT-2021::P13', 'INT-POL-ICT-07::5.2']
abstained: False


And a question the corpus can't answer, to check it abstains instead of
making something up:

In [10]:
my_question2 = "What is the minimum capital conservation buffer under CRD IV?"

answer2, _ = answer_question(retriever, my_question2)

print("Q:", my_question2)
print("A:", answer2["text"])
print("abstained:", answer2["abstained"], " <- should be True")

Q: What is the minimum capital conservation buffer under CRD IV?
A: INSUFFICIENT_EVIDENCE - none of the retrieved passages look relevant enough to answer this.
abstained: True  <- should be True


In [11]:
test_questions = [
    {"q": "What must a regulated firm notify the Central Bank about regarding outsourcing arrangements?",
     "gold": ["CBI-OUT-2021::P13"]},
    {"q": "Does the Central Bank of Ireland's outsourcing guidance prescribe a fixed number of days' notice for notifying an outsourcing arrangement?",
     "gold": ["CBI-OUT-2021::P13"]},
    {"q": "What is the deadline for the initial notification of a major ICT incident under DORA?",
     "gold": ["EU-DORA-2022::Art19"]},
    {"q": "What annual contract value triggers escalation to the Group Outsourcing Committee?",
     "gold": ["INT-POL-ICT-07::5.2"]},
    {"q": "Under what condition may a regulated firm permit someone to perform a controlled function?",
     "gold": ["CBI-FP-S21::s21"]},
    {"q": "How often should firms review the fitness and probity of controlled function holders?",
     "gold": ["CBI-FP-S21::monitoring"]},
    {"q": "What is the minimum capital conservation buffer under CRD IV?", "gold": []},
    {"q": "How many branches must the firm keep open in each county?", "gold": []},
]


def recall_at_k(retrieved_ids, gold_ids, k):
    if not gold_ids:
        return None
    top = set(retrieved_ids[:k])
    return len(top.intersection(gold_ids)) / len(gold_ids)


def mrr(retrieved_ids, gold_ids):
    if not gold_ids:
        return None
    for i, cid in enumerate(retrieved_ids, start=1):
        if cid in gold_ids:
            return 1 / i
    return 0


def average(values):
    values = [v for v in values if v is not None]
    if not values:
        return 0
    return sum(values) / len(values)


print(len(test_questions), "test questions loaded",
      f"({len([q for q in test_questions if q['gold']])} answerable, "
      f"{len([q for q in test_questions if not q['gold']])} unanswerable)")

8 test questions loaded (6 answerable, 2 unanswerable)


In [12]:
print("=== all example answers ===\n")
for item in test_questions:
    answer, _ = answer_question(retriever, item["q"])
    print("Q:", item["q"])
    print("A:", answer["text"])
    print("   cites:", answer["cites"], "  abstained:", answer["abstained"])
    print()

=== all example answers ===

Q: What must a regulated firm notify the Central Bank about regarding outsourcing arrangements?
A: Notify the Central Bank of planned critical or important outsourcing arrangements and of material changes to existing critical or important outsourcing arrangements. [1] The Central Bank has clarified that notification of such proposed arrangements does not constitute a pre-approval process and specific timings in respect of the submission of notifications are not prescribed unless required by existing regulation. [1] Any outsourcing arrangement with an annual contract value exceeding EUR 250,000 must be escalated to the Group Outsourcing Committee before contract signature. [2]
   cites: ['CBI-OUT-2021::P13', 'CBI-OUT-2021::P13', 'INT-POL-ICT-07::5.2']   abstained: False

Q: Does the Central Bank of Ireland's outsourcing guidance prescribe a fixed number of days' notice for notifying an outsourcing arrangement?
A: Notify the Central Bank of planned critical o

In [13]:
print("=== retrieval accuracy ===\n")
recalls, mrrs = [], []
for item in test_questions:
    if not item["gold"]:
        continue
    candidates = retriever.search(item["q"], top_k=10)
    top = [c for c, s in rerank(item["q"], candidates, top_n=5)]
    ids = [c["id"] for c in top]
    recalls.append(recall_at_k(ids, item["gold"], 5))
    mrrs.append(mrr(ids, item["gold"]))

print("recall@5:", round(average(recalls), 2))
print("mrr:", round(average(mrrs), 2))

=== retrieval accuracy ===

recall@5: 1.0
mrr: 1.0


In [14]:
print("=== abstention check (should all be True) ===\n")
for item in test_questions:
    if item["gold"]:
        continue
    answer, _ = answer_question(retriever, item["q"])
    print("-", item["q"], "-> abstained:", answer["abstained"])

=== abstention check (should all be True) ===

- What is the minimum capital conservation buffer under CRD IV? -> abstained: True
- How many branches must the firm keep open in each county? -> abstained: True


In [15]:
fw_chunks = filter_chunks(make_chunks_fixed_width(docs), True, True)
fw_retriever = HybridRetriever(fw_chunks)

struct_recalls, fw_recalls = [], []
for item in test_questions:
    if not item["gold"]:
        continue
    struct_top = [c["id"] for c, s in retriever.search(item["q"], top_k=5)]
    struct_recalls.append(recall_at_k(struct_top, item["gold"], 5))

    # fixed-width chunk ids don't line up with gold section ids, so check
    # whether the right *document* shows up instead
    fw_top_docs = [c["doc_id"] for c, s in fw_retriever.search(item["q"], top_k=5)]
    gold_docs = set(g.split("::")[0] for g in item["gold"])
    fw_recalls.append(1 if any(d in gold_docs for d in fw_top_docs) else 0)

print("structure-aware recall@5:", round(average(struct_recalls), 2))
print("fixed-width recall@5 (doc level):", round(average(fw_recalls), 2))

structure-aware recall@5: 1.0
fixed-width recall@5 (doc level): 1.0


In [16]:
public_chunks = filter_chunks(make_chunks(docs), allow_internal=False, current_only=True)
public_retriever = HybridRetriever(public_chunks)

q = "What annual contract value triggers escalation to the Group Outsourcing Committee?"

internal_answer, _ = answer_question(retriever, q)
public_answer, _ = answer_question(public_retriever, q)

print("Q:", q)
print()
print("internal user (sees internal policy) - abstained:", internal_answer["abstained"])
print("  ->", internal_answer["text"])
print()
print("public user (internal policy excluded) - abstained:", public_answer["abstained"], "(should be True)")
print("  ->", public_answer["text"])

Q: What annual contract value triggers escalation to the Group Outsourcing Committee?

internal user (sees internal policy) - abstained: False
  -> Any outsourcing arrangement with an annual contract value exceeding EUR 250,000 must be escalated to the Group Outsourcing Committee before contract signature. [1] Notify the Central Bank of planned critical or important outsourcing arrangements and of material changes to existing critical or important outsourcing arrangements. [2] Develop and maintain an outsourcing register to include prescribed information for all existing and future outsourcing arrangements. [3]

public user (internal policy excluded) - abstained: True (should be True)
  -> INSUFFICIENT_EVIDENCE - none of the retrieved passages look relevant enough to answer this.


In [17]:
answerable = [q for q in test_questions if q["gold"]]
random.seed(0)
random.shuffle(answerable)
half = len(answerable) // 2
tune_set = answerable[:half]
holdout_set = answerable[half:]

print("tuning on", len(tune_set), "questions, holding out", len(holdout_set))


def score_config(w_sparse, rrf_k, qset):
    r = HybridRetriever(chunks, w_sparse=w_sparse, rrf_k=rrf_k)
    recalls = []
    for item in qset:
        candidates = r.search(item["q"], top_k=10)
        top = [c for c, s in rerank(item["q"], candidates, top_n=5)]
        ids = [c["id"] for c in top]
        recalls.append(recall_at_k(ids, item["gold"], 5))
    return average(recalls)


best_score, best_config = -1, None
for w_sparse in [0.2, 0.3, 0.5, 0.7, 0.8]:
    for rrf_k in [10, 30, 60, 100]:
        score = score_config(w_sparse, rrf_k, tune_set)
        if score > best_score:
            best_score, best_config = score, (w_sparse, rrf_k)

print()
print("best config on tuning set:", best_config, " recall@5 =", round(best_score, 2))

w_sparse, rrf_k = best_config
holdout_score = score_config(w_sparse, rrf_k, holdout_set)
print("same config on held-out set: recall@5 =", round(holdout_score, 2))

tuning on 3 questions, holding out 3

best config on tuning set: (0.2, 10)  recall@5 = 1.0
same config on held-out set: recall@5 = 1.0
